In [7]:
import pandas as pd
from sqlalchemy import create_engine

# --- 1. Configuração da Conexão ---
server_name = 'srv-vibcor20'
database_name = 'PGO_STG'
engine_str = f"mssql+pyodbc://{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(engine_str)

def extrair_info_pressao(tag):
    try:
        partes = str(tag).split('.')
        if len(partes) >= 7:
            codigo = f"VRP.{partes[1]}.{partes[6]}"
            carac = 'Pressão Montante' if partes[4] == '001' else 'Pressão Jusante' if partes[4] == '002' else 'Outro'
            return pd.Series([codigo, carac])
    except: pass
    return pd.Series(['', ''])

def transformar_tag_vazao(tag):
    try:
        partes = str(tag).split('.')
        if len(partes) >= 5:
            num_vazao = int(partes[4])
            codigo = f"VZ{num_vazao}.{partes[0]}.{partes[1]}.{partes[2]}"
            return pd.Series([codigo, 'Vazão'])
    except: pass
    return pd.Series([tag, 'Vazão'])

def buscar_e_processar(tabela, tipo_dado):
    print(f"Buscando dados de {tipo_dado} na tabela {tabela}...")
    query = f"SELECT * FROM {tabela} WHERE [TEMPO] >= DATEADD(day, -7, GETDATE()) ORDER BY [TEMPO] ASC"
    
    df = pd.read_sql(query, engine)
    if df.empty:
        return pd.DataFrame()

    df['TEMPO'] = pd.to_datetime(df['TEMPO'])
    df.set_index('TEMPO', inplace=True)
    
    # Resample horário
    df_res = df.groupby('TAG').resample('h').mean(numeric_only=True).reset_index()
    
    # Aplicar lógicas específicas
    if tipo_dado == 'Pressão':
        df_res[['Código Equipamento', 'Característica']] = df_res['TAG'].apply(extrair_info_pressao)
    else:
        df_res[['Código Equipamento', 'Característica']] = df_res['TAG'].apply(transformar_tag_vazao)
    
    # Identificar a coluna de valor (assume-se que é a coluna numérica restante, ex: 'VALOR')
    # Ajuste o nome 'VALOR' se no seu banco for diferente
    col_valor = [c for c in df_res.columns if c not in ['TAG', 'TEMPO', 'Código Equipamento', 'Característica']][0]
    df_res = df_res.rename(columns={col_valor: 'Valor'})
    
    return df_res[['TAG', 'Código Equipamento', 'Característica', 'TEMPO', 'Valor']]

# --- 2. Execução ---
try:
    # Processa Pressões
    df_pres = buscar_e_processar('monitoramentoPressoes', 'Pressão')
    
    # Processa Vazões
    df_vaz = buscar_e_processar('monitoramentoVazoes', 'Vazão')

    # Unificar os DataFrames
    df_final = pd.concat([df_pres, df_vaz], ignore_index=True)

    if df_final.empty:
        print("Nenhum dado encontrado em ambas as tabelas.")
    else:
        # --- 3. Salvar em CSV ---
        nome_arquivo = "consolidado_monitoramento.csv"
        
        df_final.to_csv(
            nome_arquivo, 
            index=False, 
            sep=';', 
            encoding='utf-8-sig', 
            decimal=',', 
            float_format='%.4f'
        )

        print(f"\nSucesso! Arquivo '{nome_arquivo}' gerado com {len(df_final)} linhas.")

except Exception as e:
    print(f"Erro no processamento: {e}")

Buscando dados de Pressão na tabela monitoramentoPressoes...
Buscando dados de Vazão na tabela monitoramentoVazoes...

Sucesso! Arquivo 'consolidado_monitoramento.csv' gerado com 320424 linhas.


In [2]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Configuração da conexão
server_name = 'srv-vibcor20'
database_name = 'PGO_STG'
tabela = 'monitoramentoVazoes' # Alterado para a tabela de vazões

engine_str = f"mssql+pyodbc://{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(engine_str)

print(f"Conectando ao banco e buscando dados da última semana na tabela {tabela}...")

# 2. Query SQL
query = f"""
SELECT * FROM {tabela} 
WHERE [TEMPO] >= DATEADD(day, -7, GETDATE())
ORDER BY [TEMPO] ASC
"""

try:
    # 3. Carregar os dados
    df = pd.read_sql(query, engine)
    
    if df.empty:
        print("Nenhum dado encontrado para o período selecionado.")
    else:
        # 4. Processamento dos dados
        df['TEMPO'] = pd.to_datetime(df['TEMPO'])
        df.set_index('TEMPO', inplace=True)
        
        # Agrupa pela TAG e faz o resample por hora
        df_resampled = df.groupby('TAG').resample('h').mean(numeric_only=True)
        df_resampled.reset_index(inplace=True)
        
        # --- NOVA LÓGICA: Transformação da TAG de Vazão ---
        def transformar_tag_vazao(tag):
            try:
                partes = str(tag).split('.')
                # Exemplo: SAT.VCP.014.FIT.002.000.000
                # partes[0]=SAT, partes[1]=VCP, partes[2]=014, partes[4]=002
                
                if len(partes) >= 5:
                    # Pega o número do FIT (ex: 002 -> 2)
                    num_vazao = int(partes[4])
                    # Monta o novo formato: VZ2.SAT.VCP.014
                    nova_tag = f"VZ{num_vazao}.{partes[0]}.{partes[1]}.{partes[2]}"
                    return nova_tag
            except Exception:
                pass
            return tag # Retorna a tag original se falhar

        # Aplica a transformação diretamente na coluna TAG ou cria uma nova
        df_resampled['TAG_FORMATADA'] = df_resampled['TAG'].apply(transformar_tag_vazao)
        
        # Reorganizando as colunas para melhor visualização
        colunas = df_resampled.columns.tolist()
        colunas.remove('TAG_FORMATADA')
        idx_tag = colunas.index('TAG')
        colunas.insert(idx_tag + 1, 'TAG_FORMATADA')
        df_resampled = df_resampled[colunas]
        # --------------------------------------------------------
        
        # 5. Salvar o resultado
        nome_arquivo = "media_vazoes_ultima_semana.csv"
        
        df_resampled.to_csv(
            nome_arquivo, 
            index=False, 
            sep=';', 
            encoding='utf-8-sig', 
            decimal=',', 
            float_format='%.4f'
        )
        
        print(f"Sucesso! Arquivo '{nome_arquivo}' gerado.")
        print(f"Total de linhas processadas: {len(df_resampled)}")

except Exception as e:
    print(f"Erro ao processar os dados: {e}")

Conectando ao banco e buscando dados da última semana na tabela monitoramentoVazoes...
Sucesso! Arquivo 'media_vazoes_ultima_semana.csv' gerado.
Total de linhas processadas: 91995


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Configuração da conexão
server_name = 'srv-vibcor20'
database_name = 'PGO_STG'
tabela = 'monitoramentoPressoes'

engine_str = f"mssql+pyodbc://{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(engine_str)

print(f"Conectando ao banco e buscando dados da última semana na tabela {tabela}...")

# 2. Query SQL com filtro de tempo (Últimos 7 dias)
query = f"""
SELECT * FROM {tabela} 
WHERE [TEMPO] >= DATEADD(day, -7, GETDATE())
ORDER BY [TEMPO] ASC
"""

try:
    # 3. Carregar os dados
    df = pd.read_sql(query, engine)
    
    if df.empty:
        print("Nenhum dado encontrado para o período selecionado.")
    else:
        # 4. Processamento dos dados
        # Garante que a coluna TEMPO seja do tipo datetime
        df['TEMPO'] = pd.to_datetime(df['TEMPO'])
        
        # Define a coluna TEMPO como índice para permitir o agrupamento temporal
        df.set_index('TEMPO', inplace=True)
        
        # Agrupa primeiro pela TAG e depois faz o resample por hora ('H')
        df_resampled = df.groupby('TAG').resample('h').mean(numeric_only=True)
        
        # Reseta o índice para transformar TAG e TEMPO de volta em colunas normais no DataFrame
        df_resampled.reset_index(inplace=True)
        
        # --- NOVA LÓGICA: Extrair VRP e Característica da TAG ---
        def extrair_info_tag(tag):
            try:
                partes = str(tag).split('.')
                # Exemplo das partes separadas: ['SAT', 'VCP', '011', 'PIT', '002', 'VRP', '001']
                if len(partes) >= 7:
                    vrp = f"VRP.{partes[1]}.{partes[6]}"
                    
                    if partes[4] == '001':
                        carac = 'Montante'
                    elif partes[4] == '002':
                        carac = 'Jusante'
                    else:
                        carac = 'Outro'
                        
                    return pd.Series([vrp, carac])
            except Exception:
                pass
            return pd.Series(['', ''])
        
        # Aplica a função e cria as novas colunas
        df_resampled[['VRP', 'Característica']] = df_resampled['TAG'].apply(extrair_info_tag)
        
        # Reorganiza a ordem das colunas para colocar as novas logo após a TAG
        colunas = df_resampled.columns.tolist()
        colunas.remove('VRP')
        colunas.remove('Característica')
        idx_tag = colunas.index('TAG')
        colunas.insert(idx_tag + 1, 'VRP')
        colunas.insert(idx_tag + 2, 'Característica')
        df_resampled = df_resampled[colunas]
        # --------------------------------------------------------
        
        # 5. Salvar o resultado
        nome_arquivo = "media_pressoes_ultima_semana.csv"
        
        # index=False evita salvar os números da linha (0, 1, 2...) no CSV
        # decimal=',' ajusta o separador para o padrão do Excel em PT-BR
        # float_format='%.4f' limita as casas decimais para manter o arquivo limpo
        df_resampled.to_csv(
            nome_arquivo, 
            index=False, 
            sep=';', 
            encoding='utf-8-sig', 
            decimal=',', 
            float_format='%.4f'
        )
        
        print(f"Sucesso! Arquivo '{nome_arquivo}' gerado com as médias horárias separadas por TAG.")
        print(f"Total de linhas processadas: {len(df_resampled)}")

except Exception as e:
    print(f"Erro ao processar os dados: {e}")

In [6]:
import os
from git import Repo
# ADICIONE ESTA LINHA ABAIXO (Ajuste o caminho se o seu Git estiver em outro lugar)
PATH_REPOSITORIO = r"C:\Users\kaiorodrigues.000\Desktop\Teste-html"
ARQUIVO_EXCEL = "media_pressoes_ultima_semana.csv"
MENSAGEM_COMMIT = "Atualização diária de dados"

def subir_para_github():
    try:
        repo = Repo(PATH_REPOSITORIO)
        
        # Garante que estamos na branch correta (ex: main ou master)
        # repo.git.checkout('main') 

        # Verifica se o arquivo mudou
        if ARQUIVO_EXCEL in repo.untracked_files or repo.is_dirty(path=ARQUIVO_EXCEL):
            repo.index.add([ARQUIVO_EXCEL])
            repo.index.commit(MENSAGEM_COMMIT)
            
            origin = repo.remote(name='origin')
            origin.push()
            print(f"✅ Arquivo {ARQUIVO_EXCEL} atualizado e enviado!")
        else:
            print("ℹ️ Nenhuma alteração detectada no arquivo. Nada enviado.")

    except Exception as e:
        print(f"❌ Erro ao enviar: {e}")

if __name__ == "__main__":
    subir_para_github()

✅ Arquivo media_pressoes_ultima_semana.csv atualizado e enviado!
